In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

## BB84 Protocol Simulation with an Attacker (Eve)

This simulation demonstrates the BB84 quantum key distribution protocol with an active eavesdropper, Eve. The goal is to show how Eve's presence can be detected by Alice and Bob.

**Protocol Steps:**

1.  **Alice's Preparation:** Alice generates a random sequence of bits and chooses a random sequence of bases (rectilinear or diagonal) to encode these bits into qubits.
2.  **Eve's Attack:** Eve intercepts the qubits sent by Alice. She randomly chooses her own bases to measure the qubits. Based on her measurements, she re-encodes the qubits in her chosen bases and sends them to Bob.
3.  **Bob's Measurement:** Bob receives the qubits from Eve and randomly chooses his own bases to measure them.
4.  **Public Sifting:** Alice and Bob publicly announce their chosen bases (but not their bit values). They keep only the bits where their chosen bases matched.
5.  **Error Checking:** Alice and Bob compare a subset of their sifted keys. If the number of mismatches (Quantum Bit Error Rate - QBER) exceeds a certain threshold, they deduce the presence of an eavesdropper and discard the key. Otherwise, they use the remaining bits as their shared secret key.

In [ ]:
### Helper Functions

# Function to generate a random bit using a quantum circuit
def quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0) # Apply Hadamard to create a superposition
    qc.measure(0, 0) # Measure to collapse to 0 or 1 randomly
    simulator = BasicSimulator()
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc)
    return int(list(counts.keys())[0]) # Return 0 or 1

# Function to generate N random bits
def generate_random_bits(n):
    return [quantum_random_bit() for _ in range(n)]

# Function to encode a bit into a qubit based on a chosen basis
def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 0:
        if basis == 0: # Z-basis (rectilinear): |0>
            pass # Already in |0>
        else: # X-basis (diagonal): |+>
            qc.h(0)
    else: # bit == 1
        if basis == 0: # Z-basis (rectilinear): |1>
            qc.x(0) # Apply X to get |1>
        else: # X-basis (diagonal): |->
            qc.x(0) # Get |1>
            qc.h(0) # Apply H to get |->
    return qc # Return the circuit representing the encoded qubit

# Function to measure a qubit in a chosen basis
def measure_qubit(qc, basis):
    if basis == 0: # Z-basis
        qc.measure(0, 0)
    else: # X-basis
        qc.h(0) # Change to Z-basis for measurement
        qc.measure(0, 0)

    simulator = BasicSimulator()
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc)
    return int(list(counts.keys())[0]) # Return the measured bit

### Alice's Actions

In [ ]:
# 1. Alice generates random bits for her raw key
num_qubits = 100 # Number of qubits/bits to send
alice_bits = generate_random_bits(num_qubits)
print(f"Alice's original bits (first 10): {alice_bits[:10]}")

# 2. Alice generates random bases for encoding (0 for Z-basis, 1 for X-basis)
alice_bases = generate_random_bits(num_qubits)
print(f"Alice's encoding bases (first 10): {alice_bases[:10]}")

# 3. Alice encodes her bits into qubits based on her chosen bases
alice_qubits = []
for i in range(num_qubits):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    # We create a new circuit for each qubit so they are independent
    alice_qubits.append(qc.remove_final_measurements(inplace=False)) # Remove measurement for transmission

### Eve's Actions (Attacker)

In [ ]:
# 1. Eve intercepts Alice's qubits.

# 2. Eve generates random bases for her measurements
eve_bases = generate_random_bits(num_qubits)
print(f"Eve's measurement bases (first 10): {eve_bases[:10]}")

# 3. Eve measures the intercepted qubits and records her results
eve_measured_bits = []
eve_re_encoded_qubits = []

for i in range(num_qubits):
    # Eve performs a measurement on the qubit she received
    measured_bit = measure_qubit(alice_qubits[i], eve_bases[i])
    e_qc = encode_qubit(measured_bit, eve_bases[i]) # Re-encode based on her measurement and basis
    eve_measured_bits.append(measured_bit)
    eve_re_encoded_qubits.append(e_qc.remove_final_measurements(inplace=False)) # Prepare for sending to Bob

print(f"Eve's measured bits (first 10): {eve_measured_bits[:10]}")
print(f"Eve has re-encoded and sent {len(eve_re_encoded_qubits)} qubits to Bob.")

### Bob's Actions

In [ ]:
# 1. Bob generates random bases for his measurements
bob_bases = generate_random_bits(num_qubits)
print(f"Bob's measurement bases (first 10): {bob_bases[:10]}")

# 2. Bob measures the received (from Eve) qubits based on his chosen bases
bob_measured_bits = []
for i in range(num_qubits):
    measured_bit = measure_qubit(eve_re_encoded_qubits[i], bob_bases[i])
    bob_measured_bits.append(measured_bit)

print(f"Bob's measured bits (first 10): {bob_measured_bits[:10]}")

### Key Sifting

In [ ]:
# Alice and Bob publicly compare their bases and keep only the bits where bases matched
alice_sifted_key = []
bob_sifted_key = []
matching_indices = []

for i in range(num_qubits):
    if alice_bases[i] == bob_bases[i]:
        alice_sifted_key.append(alice_bits[i])
        bob_sifted_key.append(bob_measured_bits[i])
        matching_indices.append(i)

print(f"Number of matching bases: {len(alice_sifted_key)}")
print(f"Alice's sifted key (first 10): {alice_sifted_key[:10]}")
print(f"Bob's sifted key (first 10): {bob_sifted_key[:10]}")

### Error Checking

In [ ]:
# Alice and Bob compare a subset of their sifted keys to detect Eve
check_key_length = min(len(alice_sifted_key), len(bob_sifted_key)) // 2 # Use half of the sifted key for checking

# Split the sifted keys into a checking portion and the final key portion
alice_checking_key = alice_sifted_key[:check_key_length]
bob_checking_key = bob_sifted_key[:check_key_length]

alice_final_key = alice_sifted_key[check_key_length:]
bob_final_key = bob_sifted_key[check_key_length:]

# Compare the checking keys
mismatches = 0
for i in range(len(alice_checking_key)):
    if alice_checking_key[i] != bob_checking_key[i]:
        mismatches += 1

error_rate = mismatches / len(alice_checking_key) if len(alice_checking_key) > 0 else 0

print(f"\n--- Protocol Summary ---")
print(f"Total qubits sent: {num_qubits}")
print(f"Length of sifted key: {len(alice_sifted_key)}")
print(f"Length of checking key: {len(alice_checking_key)}")
print(f"Mismatches in checking key: {mismatches}")
print(f"Quantum Bit Error Rate (QBER): {error_rate:.2%}")

# Define a threshold for detecting an attacker
threshold = 0.25 # 25% error rate is a common threshold

if error_rate > threshold:
    print(f"QBER ({error_rate:.2%}) is above the threshold ({threshold:.2%}). Attack detected! Key discarded.")
else:
    print(f"QBER ({error_rate:.2%}) is below the threshold ({threshold:.2%}). No attack detected (or Eve was lucky). Key can be used.")
    print(f"Alice's final shared key (first 10): {alice_final_key[:10]}")
    print(f"Bob's final shared key (first 10): {bob_final_key[:10]}")

# Verify if the final keys match (they should if no attack or error rate is low enough)
if alice_final_key == bob_final_key:
    print("Final keys match (after error checking and sifting). Perfect key established.")
else:
    print("Final keys DO NOT match, even after error checking. The remaining key is compromised.")